# SOC-Triage-Gym v2 — GRPO Training

**Hackathon: OpenEnv Apr 2026 — Theme #1 Multi-Agent Interactions + Fleet AI Oversight**

This notebook trains a Tier-1 SOC Analyst LLM on multi-agent team episodes using **GRPO** (Group Relative Policy Optimization).

### Environment Architecture
```
Alert Queue → [Tier-1 Analyst] → ESCALATE_TO_TIER2 → [Tier-2 Responder] → [SOC Manager]
               (40 steps)           (ticket bus)        (20 steps)          (8 steps)
                                                                    ↑
                                                         [Red-Team Generator]
                                                         (adaptive curriculum)
```

### Reward Structure
- **Per step**: `0.6 × role_specific + 0.4 × team_F1`
- **Episode final**: `0.6 × individual + 0.4 × team_F1` (prevents GRPO advantage collapse)

### Training Plan
| Phase | Role | Frozen Roles | GPU Time |
|-------|------|-------------|----------|
| Day 1 AM | Tier-1 | Scripted T2+Manager | ~4h H100 |
| Day 1 PM | Tier-2 | Trained T1 | ~3h H100 |
| Day 2 AM | Manager | Trained T1+T2 | ~2h H100 |
| Day 2 PM | Joint | All co-trained | ~4h H100 |

## 1. Install Dependencies

In [ ]:
%%capture
# Core dependencies
!pip install unsloth trl>=0.9.0 peft transformers accelerate datasets
!pip install fastapi uvicorn httpx pydantic
!pip install matplotlib numpy

# Clone the repo (replace with your HF Space URL or local path)
import os
if not os.path.exists('openenv-2'):
    !git clone https://github.com/rohitcraftsyt/openenv-2.git openenv-2
%cd openenv-2

## 2. Start SOC-Triage-Gym Server

In [ ]:
import subprocess, time, httpx

# Start the FastAPI server in background
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# Verify it's healthy
client = httpx.Client(base_url='http://localhost:7860', timeout=30.0)
resp = client.get('/health')
print(f'Server status: {resp.status_code} — {resp.json()}')

## 3. Verify Environment — Run Sample Episode

In [ ]:
import json, httpx

client = httpx.Client(base_url='http://localhost:7860', timeout=30.0)

# Test 1: Legacy solo mode (backward compat)
obs = client.post('/reset', json={'task_id': 'phishing', 'seed': 42}).json()
print(f"Solo mode — alerts: {len(obs['alert_queue'])}, mode: {obs.get('episode_mode')}")

# Test 2: Team mode
obs = client.post('/reset', json={'task_id': 'team_phishing_escalation', 'seed': 42, 'mode': 'team'}).json()
print(f"Team mode — alerts: {len(obs['alert_queue'])}, role: {obs.get('current_role')}, phase: {obs.get('current_phase')}")

# Test 3: Red-team generated scenario
gen_resp = client.post('/generate_scenario', json={'seed': 42})
gen_data = gen_resp.json()
print(f"Generated scenario — task_id: {gen_data.get('task_id')}, difficulty: {gen_data.get('difficulty_floor')}")

print('\n✓ Environment verified')

## 4. Baseline: Oracle Heuristic Reward Curves

Run the scripted oracle agent to establish baseline scores before training.

In [ ]:
import sys
sys.path.insert(0, '.')
from train_grpo import run_episode, oracle_action, TIER1_TASKS, SEEDS

baseline_scores = []
print('Running baseline oracle episodes...')
for task_id in TIER1_TASKS:
    for seed in SEEDS[:10]:  # 10 seeds per task for baseline
        score, traj = run_episode(client, task_id=task_id, seed=seed, role_to_train='tier1')
        baseline_scores.append({'task': task_id, 'seed': seed, 'score': score})
        print(f'  {task_id} seed={seed}: {score:.4f}')

baseline_avg = sum(s['score'] for s in baseline_scores) / len(baseline_scores)
print(f'\nOracle baseline avg: {baseline_avg:.4f}')

## 5. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import os

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # Swap for larger: Qwen2.5-7B-Instruct
HF_TOKEN = os.getenv('HF_TOKEN', '')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=768,
    dtype=None,          # auto-detect bf16/fp16
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()

## 6. Build Training Dataset from Environment

In [ ]:
from datasets import Dataset
from train_grpo import build_prompt_dataset, ROLE_SYSTEM_PROMPTS

ROLE = 'tier1'  # Change to 'tier2' or 'manager' for staged training
TRAIN_SEEDS = list(range(42, 42 + 40))  # 40 seeds per task

print(f'Building prompt dataset for role={ROLE}...')
raw_data = build_prompt_dataset(client, TIER1_TASKS, TRAIN_SEEDS, ROLE)

import random
random.seed(42)
random.shuffle(raw_data)

split = max(1, len(raw_data) // 10)
train_data = raw_data[split:]
eval_data = raw_data[:split]

train_dataset = Dataset.from_list([{'prompt': d['prompt'], 'task_id': d['task_id'], 'seed': d['seed']} for d in train_data])
print(f'Train: {len(train_dataset)} | Eval: {len(eval_data)}')

## 7. Define Reward Function

In [ ]:
from train_grpo import make_reward_fn

reward_fn = make_reward_fn(client, ROLE)

# Quick sanity check
test_completions = [[
    {'role': 'assistant', 'content': json.dumps({
        'action_type': 'enrich_indicator',
        'indicator': '203.57.1.1',
        'indicator_type': 'ip',
        'query_alert_id': 'ALT-TEAM-001',
        'role': 'tier1'
    })}
]]
test_rewards = reward_fn(
    prompts=['test'],
    completions=test_completions,
    task_id=['team_phishing_escalation'],
    seed=[42]
)
print(f'Reward sanity check: {test_rewards}')

## 8. GRPO Training (group=8)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir=f'./soc_grpo_{ROLE}',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    num_generations=8,               # GRPO group size (plan: group=8)
    max_prompt_length=512,
    max_completion_length=256,
    learning_rate=5e-6,
    beta=0.04,                        # KL penalty
    temperature=1.0,
    logging_steps=5,
    save_steps=50,
    report_to=['none'],
    run_name=f'soc-grpo-{ROLE}',
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Training {len(train_dataset)} examples × 3 epochs, GRPO group=8')
trainer.train()

## 9. Save Model

In [ ]:
output_path = f'./soc_grpo_{ROLE}'
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
print(f'Model saved to {output_path}')

# Optional: push to HuggingFace Hub
# model.push_to_hub(f'YOUR_HF_USERNAME/soc-triage-gym-{ROLE}', token=HF_TOKEN)

## 10. Post-Training Evaluation — Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from unsloth import FastLanguageModel
from train_grpo import parse_action_from_text, format_obs_prompt

FastLanguageModel.for_inference(model)

def run_trained_episode(task_id, seed):
    """Run a team episode where the trained model handles the target role."""
    obs = client.post('/reset', json={'task_id': task_id, 'seed': seed, 'mode': 'team'}).json()
    step = 0
    while not obs.get('done', False) and step < 80:
        step += 1
        role = obs.get('current_role', 'tier1')
        if role == ROLE:
            prompt_text = format_obs_prompt(obs, role, step)
            messages = [
                {'role': 'system', 'content': ROLE_SYSTEM_PROMPTS[ROLE]},
                {'role': 'user', 'content': prompt_text}
            ]
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
            ).to(model.device)
            outputs = model.generate(
                inputs, max_new_tokens=128, temperature=0.0, do_sample=False
            )
            text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            action = parse_action_from_text(text, role)
        else:
            action = oracle_action(obs)
        step_resp = client.post('/step', content=json.dumps(action),
                                headers={'Content-Type': 'application/json'})
        obs = step_resp.json()
    return obs.get('task_score') or obs.get('cumulative_reward', 0.0)

# Evaluate on held-out seeds
EVAL_SEEDS = list(range(100, 115))  # seeds not seen during training
trained_scores = []
print('Evaluating trained model...')
for task_id in TIER1_TASKS:
    for seed in EVAL_SEEDS:
        score = run_trained_episode(task_id, seed)
        trained_scores.append(score)
        print(f'  {task_id} seed={seed}: {score:.4f}')

trained_avg = sum(trained_scores) / len(trained_scores)
print(f'\nTrained model avg: {trained_avg:.4f}')
print(f'Oracle baseline avg: {baseline_avg:.4f}')
print(f'Improvement: {trained_avg - baseline_avg:+.4f} ({(trained_avg/baseline_avg - 1)*100:+.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SOC-Triage-Gym v2 — GRPO Training Results', fontsize=14, fontweight='bold')

# Left: before/after comparison
ax = axes[0]
ax.bar(['Oracle Baseline', f'GRPO Trained\n(role={ROLE})'],
       [baseline_avg, trained_avg],
       color=['#4C72B0', '#DD8452'], alpha=0.85, edgecolor='black')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Episode Score')
ax.set_title('Before vs After Training')
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='0.5 threshold')
for i, v in enumerate([baseline_avg, trained_avg]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Right: per-episode scores
ax2 = axes[1]
bl_scores_eval = baseline_scores[:len(EVAL_SEEDS) * len(TIER1_TASKS)]
bl_vals = [s['score'] for s in bl_scores_eval[:len(trained_scores)]]
x = range(1, len(trained_scores) + 1)
ax2.plot(x, bl_vals, 'o-', alpha=0.5, label='Oracle baseline', color='#4C72B0')
ax2.plot(x, trained_scores, 's-', alpha=0.7, label=f'GRPO trained ({ROLE})', color='#DD8452')

# Smoothed lines
w = min(5, len(trained_scores))
if len(trained_scores) >= w:
    sm = np.convolve(trained_scores, np.ones(w)/w, mode='valid')
    ax2.plot(range(w, len(trained_scores)+1), sm, linewidth=2.5, color='#C44E52', label='Trained (smoothed)')

ax2.set_xlabel('Evaluation Episode')
ax2.set_ylabel('Score')
ax2.set_title('Per-Episode Scores (Held-Out Seeds)')
ax2.set_ylim(0, 1.0)
ax2.legend()

plt.tight_layout()
plt.savefig('soc_grpo_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: soc_grpo_results.png')

## 11. Red-Team Curriculum — Blue Win Rate Oscillation

This is the **Theme #4 Self-Improvement money shot**: blue-team win rate oscillates around 0.5 as the Red-Team Generator adapts difficulty.

In [ ]:
from scenarios.red_team_generator import RedTeamGenerator
from models import RedTeamConfig
import matplotlib.pyplot as plt

# Simulate curriculum: N rounds of difficulty adaptation
NUM_ROUNDS = 15
EPISODES_PER_ROUND = 5

generator = RedTeamGenerator(seed=42)
difficulty_history = []
blue_win_rate_history = []

print('Running Red-Team curriculum simulation...')
for round_num in range(NUM_ROUNDS):
    round_scores = []
    for seed_offset in range(EPISODES_PER_ROUND):
        # Generate scenario
        gen_resp = client.post('/generate_scenario', json={
            'difficulty_floor': generator.config.difficulty_floor,
            'noise_density': generator.config.noise_density,
            'seed': generator.seed + seed_offset
        })
        if gen_resp.status_code != 200:
            round_scores.append(0.5)
            continue
        # Blue team plays (oracle heuristic)
        obs = client.post('/reset', json={'task_id': 'red_team_generated', 'seed': generator.seed + seed_offset, 'mode': 'team'}).json()
        step = 0
        while not obs.get('done', False) and step < 50:
            step += 1
            action = oracle_action(obs)
            obs = client.post('/step', content=json.dumps(action),
                             headers={'Content-Type': 'application/json'}).json()
        score = obs.get('task_score') or obs.get('cumulative_reward', 0.0)
        round_scores.append(float(score))

    blue_win_rate = sum(1 for s in round_scores if s > 0.5) / len(round_scores)
    diff = generator.config.difficulty_floor

    difficulty_history.append(diff)
    blue_win_rate_history.append(blue_win_rate)

    print(f'  Round {round_num+1:2d} | difficulty={diff:.2f} | blue_win_rate={blue_win_rate:.2f} | scores={[f"{s:.2f}" for s in round_scores]}')

    # Adapt difficulty for next round
    generator = generator.adapt_difficulty(blue_win_rate)

# Plot the oscillation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('SOC-Triage-Gym v2 — Red-Team Curriculum (Theme #4 Self-Improvement)', fontweight='bold')

rounds = range(1, NUM_ROUNDS + 1)
ax1.plot(rounds, blue_win_rate_history, 'o-', color='#2196F3', linewidth=2, label='Blue win rate')
ax1.axhline(0.5, linestyle='--', color='red', alpha=0.7, label='Target 0.5')
ax1.fill_between(rounds, 0.4, 0.6, alpha=0.1, color='green', label='Sweet spot [0.4, 0.6]')
ax1.set_ylabel('Blue Win Rate')
ax1.set_ylim(0, 1)
ax1.legend()
ax1.set_title('Blue-Team Win Rate (oscillates around 0.5 as Red-Team adapts)')

ax2.plot(rounds, difficulty_history, 's-', color='#F44336', linewidth=2, label='Difficulty floor')
ax2.set_xlabel('Round')
ax2.set_ylabel('Difficulty Floor')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.set_title('Red-Team Difficulty Adaptation (escalates when blue wins too often)')

plt.tight_layout()
plt.savefig('redteam_curriculum.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: redteam_curriculum.png')

## 12. Summary

### What we demonstrated

| Metric | Value |
|--------|-------|
| Oracle baseline avg score | `baseline_avg` |
| GRPO trained model avg score | `trained_avg` |
| Reward improvement | `trained_avg - baseline_avg` |
| Tasks trained on | team_phishing_escalation, team_lateral_team |
| Agent roles | Tier-1, Tier-2, SOC Manager (Fleet AI Oversight) |
| Red-Team rounds | 15 (curriculum oscillates around 0.5) |

### Theme coverage
- **Theme #1 Multi-Agent Interactions**: T1 → T2 → Manager team with inter-agent ticket bus
- **Fleet AI Scalable Oversight**: SOC Manager monitors, audits, and explains Tier-1/Tier-2 decisions
- **Theme #4 Self-Improvement**: Red-Team Generator co-evolves difficulty; see oscillation curve above

### Files
- `soc_grpo_results.png` — before/after training reward comparison
- `redteam_curriculum.png` — blue win rate oscillation (Theme #4 money shot)
- `./soc_grpo_tier1/` — trained Tier-1 LoRA weights